In [2]:
from datasets import concatenate_datasets, load_dataset
from datasets.exceptions import DatasetNotFoundError

task_names = [
    "task075_squad1.1_answer_generation",
    "task195_sentiment140_classification",
    "task568_circa_question_generation",
    "task403_creak_commonsense_inference",
    "task1358_xlsum_title_generation",
]

# Canonical repo id is hyphenated; underscore form is fallback.
dataset_candidates = [
    "Muennighoff/natural-instructions",
]

last_error = None
full_dataset = None
for dataset_id in dataset_candidates:
    try:
        split_parts = []
        for split_name in ("train", "test"):
            try:
                part = load_dataset(
                    dataset_id,
                    split=split_name,
                    verification_mode="no_checks",
                )
                split_parts.append(part)
                print(f"Loaded split '{split_name}' from {dataset_id}: {len(part)} rows")
            except (DatasetNotFoundError, ValueError):
                pass

        if split_parts:
            full_dataset = (
                concatenate_datasets(split_parts)
                if len(split_parts) > 1
                else split_parts[0]
            )
            break
    except (DatasetNotFoundError, ValueError) as err:
        last_error = err

if full_dataset is None:
    raise RuntimeError(
        f"Could not load any known dataset id: {dataset_candidates}. Last error: {last_error}"
    )

# Task ids are stored as data fields, not builder configs.
task_key_candidates = ["task_name", "task", "Task", "id", "task_id"]
task_key = next((k for k in task_key_candidates if k in full_dataset.column_names), None)

if task_key is None:
    raise RuntimeError(
        f"Could not find a task column. Available columns: {full_dataset.column_names}"
    )

raw_datasets = {}
for task in task_names:
    task_subset = full_dataset.filter(lambda ex: ex[task_key] == task)
    if len(task_subset) == 0 and task_key == "id":
        task_subset = full_dataset.filter(
            lambda ex: isinstance(ex["id"], str) and ex["id"].startswith(task)
        )
    raw_datasets[task] = task_subset

loaded_non_empty = sum(len(ds) > 0 for ds in raw_datasets.values())
print(f"Loaded {loaded_non_empty}/{len(task_names)} tasks with at least one example.")

c:\Users\abbhi\anaconda3\envs\CIS4930\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded split 'train' from Muennighoff/natural-instructions: 3082094 rows
Loaded split 'test' from Muennighoff/natural-instructions: 484006 rows
Loaded 5/5 tasks with at least one example.


In [3]:
# testing purposes

from collections import Counter

# Show which requested tasks are missing and suggest close matches.
counts = Counter(full_dataset[task_key])

print(f"task key in use: {task_key}")
print("\nRequested task counts:")
for t in task_names:
    print(f"- {t}: {counts.get(t, 0)}")

missing = [t for t in task_names if counts.get(t, 0) == 0]
print(f"\nMissing tasks ({len(missing)}): {missing}")

if missing:
    all_tasks = list(counts.keys())
    print("\nClosest available task ids:")
    import difflib

    for t in missing:
        close = difflib.get_close_matches(t, all_tasks, n=5, cutoff=0.45)
        print(f"\n{t}")
        for c in close:
            print(f"  -> {c} (count={counts[c]})")

task key in use: task_name

Requested task counts:
- task075_squad1.1_answer_generation: 6983
- task195_sentiment140_classification: 6500
- task568_circa_question_generation: 6500
- task403_creak_commonsense_inference: 6008
- task1358_xlsum_title_generation: 6127

Missing tasks (0): []


In [4]:
import random

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
CLIENTS_PER_GROUP = 5

TASK_GROUPS = {
    "qa": ["task075_squad1.1_answer_generation"],
    "summarization": ["task1358_xlsum_title_generation"],
    "classification": ["task195_sentiment140_classification"],
    "question generation": ["task568_circa_question_generation"],
    "commonsense inference": ["task403_creak_commonsense_inference"],
}


In [21]:
def high_hetero_split(tokenized_tasks, task_groups, clients_per_group=2):
    total_clients = clients_per_group * len(task_groups)
    print(f"Total clients: {total_clients} ({clients_per_group} per group × {len(task_groups)} groups)")

    task_type_map = {task: ttype for ttype, tasks in task_groups.items() for task in tasks}
    client_data = []
    assignments = []  # (example_id, client_id)

    client_id = 0
    for group_name, tasks in task_groups.items():
        # Gather all samples for this group
        group_samples = []
        for task in tasks:
            if task in tokenized_tasks:
                for i in range(len(tokenized_tasks[task])):
                    group_samples.append((task, i))

        random.shuffle(group_samples)
        chunks = [group_samples[i::clients_per_group] for i in range(clients_per_group)]

        for chunk in chunks:
            client_data.append(chunk)
            for (task, idx) in chunk:
                assignments.append({
                    "example_id": f"{task}_{idx}",
                    "client_id": client_id,
                    "heterogeneity_level": "high"
                })
            client_id += 1
    return client_data

In [22]:
def low_hetero_split(tokenized_tasks, num_clients):
    from collections import defaultdict

    # Group samples by task
    task_buckets = {}
    for task_name, ds in tokenized_tasks.items():
        samples = [(task_name, i) for i in range(len(ds))]
        random.shuffle(samples)
        task_buckets[task_name] = samples

    # Distribute each task's samples round-robin across all clients
    client_data = [[] for _ in range(num_clients)]
    for task_name, samples in task_buckets.items():
        for idx, sample in enumerate(samples):
            client_data[idx % num_clients].append(sample)

    for i, cd in enumerate(client_data):
        task_counts = Counter(s[0] for s in cd)
        # Each client should have ~equal counts of each task
    
    return client_data

In [23]:
sample = raw_datasets["task075_squad1.1_answer_generation"][0]
print("Keys:", sample.keys())
print()
for k, v in sample.items():
    print(f"{k}: {repr(v)[:100]}")  # truncate long fields

Keys: dict_keys(['task_name', 'id', 'definition', 'inputs', 'targets'])

task_name: 'task075_squad1.1_answer_generation'
id: 'task075-592460b351c247c4ad80960adce72716'
definition: 'This task is about writing a correct answer for the reading comprehension task. Based on the inform
inputs: 'Passage: While the Armenian Apostolic Church remains the most prominent church in the Armenian comm
targets: 'Armenian Apostolic Church'


In [24]:
import sqlite3, os

os.makedirs("data/processed", exist_ok=True)
DB_PATH = "data/processed/federated_data.db"

def init_db(db_path):
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.executescript("""
        CREATE TABLE IF NOT EXISTS task_manifest (
            task_name         TEXT PRIMARY KEY,
            task_type       TEXT NOT NULL,
            definition      TEXT,
            samples_target  INTEGER
        );

        CREATE TABLE IF NOT EXISTS examples (
            example_id  TEXT PRIMARY KEY,
            task_name     TEXT NOT NULL,
            input       TEXT,
            target      TEXT,
            FOREIGN KEY (task_name) REFERENCES task_manifest(task_name)
        );

        CREATE TABLE IF NOT EXISTS client_assignments (
            example_id          TEXT NOT NULL,
            client_id           INTEGER NOT NULL,
            heterogeneity_level TEXT CHECK(heterogeneity_level IN ('low', 'high')),
            PRIMARY KEY (example_id, heterogeneity_level),
            FOREIGN KEY (example_id) REFERENCES examples(example_id)
        );
    """)
    con.commit()
    return con

# ── Populate task_manifest ────────────────────────────────────────────────────
def insert_task_manifest(con, raw_datasets, task_type_map, samples_target=500):
    cur = con.cursor()
    for task, ds in raw_datasets.items():
        definition = ds[0]["definition"] if len(ds) > 0 else ""
        cur.execute("""
            INSERT OR REPLACE INTO task_manifest 
            VALUES (?, ?, ?, ?)
        """, (task, task_type_map.get(task, "unknown"), definition, samples_target))
    con.commit()
    print(f"✓ task_manifest: {len(raw_datasets)} rows inserted")

# ── Populate examples ─────────────────────────────────────────────────────────
def insert_examples(con, raw_datasets):
    cur = con.cursor()
    rows = []
    for task, ds in raw_datasets.items():
        n = len(ds)
        for i, ex in enumerate(ds):
            target = ex["targets"]
            if isinstance(target, list):
                target = target[0] if target else ""
            rows.append((ex["id"], task, ex["inputs"], target))
    
    cur.executemany("""
        INSERT OR REPLACE INTO examples VALUES (?, ?, ?, ?)
    """, rows)
    con.commit()
    print(f"✓ examples: {len(rows)} rows inserted")

# ── Populate client_assignments ───────────────────────────────────────────────
def insert_client_assignments(con, client_data, heterogeneity_level):
    cur = con.cursor()
    rows = []
    for client_id, samples in enumerate(client_data):
        for (task, idx) in samples:
            # Need the actual example_id — look it up from what we stored
            rows.append((f"{task}_{idx}", client_id, heterogeneity_level))
    
    cur.executemany("""
        INSERT OR REPLACE INTO client_assignments VALUES (?, ?, ?)
    """, rows)
    con.commit()
    print(f"✓ client_assignments ({heterogeneity_level}): {len(rows)} rows inserted")

In [25]:
# ── Run everything ────────────────────────────────────────────────────────────

high_clients = high_hetero_split(raw_datasets, TASK_GROUPS, clients_per_group=CLIENTS_PER_GROUP)
low_clients = low_hetero_split(raw_datasets, num_clients=CLIENTS_PER_GROUP * len(TASK_GROUPS))
con = init_db(DB_PATH)
insert_task_manifest(con, raw_datasets, TASK_GROUPS)
insert_examples(con, raw_datasets)
insert_client_assignments(con, high_clients, "high")
insert_client_assignments(con, low_clients, "low")
con.close()

print(f"\n✓ Database written to {DB_PATH}")

Total clients: 25 (5 per group × 5 groups)
✓ task_manifest: 5 rows inserted
✓ examples: 32118 rows inserted
✓ client_assignments (high): 32118 rows inserted
✓ client_assignments (low): 32118 rows inserted

✓ Database written to data/processed/federated_data.db
